Dado que el entrenamiento de redes neuronales es una tarea  muy costosa, **se recomienda ejecutar el notebooks en [Google Colab](https://colab.research.google.com)**, por supuesto también se puede ejecutar en local.

Al entrar en [Google Colab](https://colab.research.google.com) bastará con hacer click en `upload` y subir este notebook. No olvide luego descargarlo en `File->Download .ipynb`

**El examen deberá ser entregado con las celdas ejecutadas, si alguna celda no está ejecutadas no se contará.**

El examen se divide en tres partes, con la puntuación que se indica a continuación. La puntuación máxima será 10.

    
- [Actividad 1: Redes Recurrentes](#actividad_1): 10 pts
    - [Cuestión 1](#3.1): 2.5 pt
    - [Cuestión 2](#3.2): 2.5 pt
    - [Cuestión 3](#3.3): 2.5 pts
    - [Cuestión 4](#3.4): 1.25 pts
    - [Cuestión 5](#3.5): 1.25 pts



In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

2026-05-11 19:13:21.048481: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-11 19:13:21.054827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-11 19:13:21.062840: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-11 19:13:21.064990: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-11 19:13:21.071189: I tensorflow/core/platform/cpu_feature_guar

<a name='actividad_1'></a>
# Actividad 1: Redes Recurrentes


- [Cuestión 1](#3.1): 2.5 pt
- [Cuestión 2](#3.2): 2.5 pt
- [Cuestión 3](#3.3): 2.5 pts
- [Cuestión 4](#3.4): 1.25 pts
- [Cuestión 5](#3.5): 1.25 pts

Vamos a usar un dataset de las temperaturas mínimas diarias en Melbourne. La tarea será la de predecir la temperatura mínima en dos días. Puedes usar técnicas de series temporales vistas en otras asignaturas, pero no es necesario.


In [2]:
dataset_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'
data_dir = tf.keras.utils.get_file('daily-min-temperatures.csv', origin=dataset_url)

In [3]:
df = pd.read_csv(data_dir, parse_dates=['Date'])
df.head()

,Date,Temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [4]:
temperatures = df['Temp'].values
print('number of samples:', len(temperatures))
train_data = temperatures[:3000]
test_data = temperatures[3000:]
print('number of train samples:', len(train_data))
print('number of test samples:', len(test_data))
print('firsts trainn samples:', train_data[:10])

number of samples: 3650
number of train samples: 3000
number of test samples: 650
firsts trainn samples: [20.7 17.9 18.8 14.6 15.8 15.8 15.8 17.4 21.8 20. ]


<a name='3.1'></a>
## Cuestión 1: Convierta `train_data` y `test_data`  en ventanas de tamaño 5, para predecir el valor en 2 días

En la nomenclatura de [Introduction_to_RNN_Time_Series.ipynb](https://github.com/ezponda/intro_deep_learning/blob/main/class/RNN/Introduction_to_RNN_Time_Series.ipynb)
```python
past, future = (5, 2)
```

Para las primeras 10 muestras de train_data `[20.7, 17.9, 18.8, 14.6, 15.8, 15.8, 15.8, 17.4, 21.8, 20. ]` el resultado debería ser:

```python
x[0] : [20.7, 17.9, 18.8, 14.6, 15.8] , y[0]: 15.8
x[1] : [17.9, 18.8, 14.6, 15.8, 15.8] , y[1]: 17.4
x[2] : [18.8, 14.6, 15.8, 15.8, 15.8] , y[2]: 21.8
x[3] : [14.6, 15.8, 15.8, 15.8, 17.4] , y[3]: 20.             
```

In [5]:
# windowing function

def create_windows_np(data, window_size, horizon, shuffle=False):
    """
    Creates a dataset from the given time series data using NumPy.

    Parameters:
    data (np.ndarray): Time series data with one dimension.
    window_size (int): The number of past time steps to use as input features.
    horizon (int): The number of future time steps to predict.
    shuffle (bool): Shuffle the windows or not.

    Returns:
    tuple: A tuple containing the input-output pairs (windows, targets) as NumPy arrays.
    """

    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size+horizon-1])

    X, y = np.array(X), np.array(y)

    if shuffle:
        indices = np.arange(len(X))
        np.random.shuffle(indices)
        X, y = X[indices], y[indices]
    
    return X, y



In [6]:
past, future = (5, 2)
X_train, y_train = create_windows_np(train_data,
                                     window_size=past,
                                     horizon=future,
                                     shuffle=False)

X_test, y_test = create_windows_np(test_data,
                                     window_size=past,
                                     horizon=future,
                                     shuffle=False)

print(X_train, y_train)

for i in range(4):
    print(f'x[{i}] : {X_train[i].tolist()} , y[{i}]: {y_train[i]}')

[[20.7 17.9 18.8 14.6 15.8]
 [17.9 18.8 14.6 15.8 15.8]
 [18.8 14.6 15.8 15.8 15.8]
 ...
 [12.8 15.1 15.  14.8 12. ]
 [15.1 15.  14.8 12.  12.5]
 [15.  14.8 12.  12.5 15. ]] [15.8 17.4 21.8 ... 15.  17.1 17.3]
x[0] : [20.7, 17.9, 18.8, 14.6, 15.8] , y[0]: 15.8
x[1] : [17.9, 18.8, 14.6, 15.8, 15.8] , y[1]: 17.4
x[2] : [18.8, 14.6, 15.8, 15.8, 15.8] , y[2]: 21.8
x[3] : [14.6, 15.8, 15.8, 15.8, 17.4] , y[3]: 20.0


<a name='3.2'></a>
## Cuestión 2: Cree un modelo recurrente de dos capas GRU para predecir con las ventanas de la cuestión anterior.


In [7]:
window_size = past
inputs_shape = (window_size, 1)
inputs = keras.layers.Input(shape=inputs_shape)

x = tf.keras.layers.GRU(units=128, return_sequences=True)(inputs)
x = tf.keras.layers.GRU(units=64)(x)
outputs = tf.keras.layers.Dense(1)(x)

model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='mse')
model.summary()

I0000 00:00:1778519602.501418   56927 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-11 19:13:22.523518: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2343] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 5, 1)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 5, 128)         │        50,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 87,617 (342.25 KB)

 Trainable params: 87,617 (342.25 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=10)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2, shuffle=True, batch_size = 64, callbacks=[es_callback]
)

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 40.3013 - val_loss: 17.6639
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 16.2678 - val_loss: 16.5206
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 14.0266 - val_loss: 13.8034
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 11.7643 - val_loss: 11.0801
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10.2511 - val_loss: 9.5466
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 9.5556 - val_loss: 9.4258
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9.1116 - val_loss: 8.6646
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9.1233 - val_loss: 9.2165
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 8.8895 - val_loss: 8.9336
Epoch 10/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 8.8298 - val_loss: 8.4161
Epoch 11/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 8.7998 - val_loss: 8.3826
Epoch 12/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - 

In [9]:
results = model.evaluate(X_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 7.0404 
Test Loss: 7.040409088134766


<a name='3.3'></a>
## Cuestión 3: Añada más features a la series temporal, por ejemplo `portion_year`. Cree un modelo que mejore al anterior.


In [10]:
# 0. Se define la función para múltiples features
def create_windows_multivariate_np(data, window_size, horizon, target_col_idx, shuffle=False):
    """
    Creates a dataset from the given time series data using NumPy.

    Parameters:
    data (np.ndarray or pd.DataFrame): Time series data with multiple features.
    window_size (int): The number of past time steps to use as input features.
    horizon (int): The number of future time steps to predict.
    target_col_idx (int): The index of the target column in the input data.
    shuffle (bool): Whether to shuffle the data or not.

    Returns:
    tuple: A tuple containing the input-output pairs (X, y) as NumPy arrays.
    """
    if isinstance(data, pd.DataFrame):
        data = data.values

    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i:i+window_size, :])
        y.append(data[i+window_size+horizon-1, target_col_idx])

    X, y = np.array(X), np.array(y)

    if shuffle:
        indices = np.arange(X.shape[0])
        np.random.shuffle(indices)
        X, y = X[indices], y[indices]

    return X, y

In [11]:
# 1. Codificación cíclica (sin/cos) para día del año y mes
df['day_sin']   = np.sin(2 * np.pi * df['Date'].dt.dayofyear / 365.0)
df['day_cos']   = np.cos(2 * np.pi * df['Date'].dt.dayofyear / 365.0)
df['month_sin'] = np.sin(2 * np.pi * df['Date'].dt.month / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['Date'].dt.month / 12.0)

# 2. Media móvil de los últimos 7 días (tendencia local reciente)
df['temp_roll7'] = df['Temp'].rolling(window=7, min_periods=1).mean()

df_improved = df[['Temp', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'temp_roll7']].copy()

# 3. Se estandarizan las features (solo en train, luego se aplica la misma transformación al test)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_imp = scaler.fit_transform(df_improved.iloc[:3000].values)
test_imp  = scaler.transform(df_improved.iloc[3000:].values)

In [12]:
# 4. Se crean las ventanas como antes
X_train_imp, y_train_imp = create_windows_multivariate_np(
    train_imp, window_size=past, horizon=future, target_col_idx=0)
X_test_imp,  y_test_imp  = create_windows_multivariate_np(
    test_imp,  window_size=past, horizon=future, target_col_idx=0)

print('X_train shape:', X_train_imp.shape)

X_train shape: (2994, 5, 6)


In [13]:
# 5. Se añade Dropout para regularización
n_features = X_train_imp.shape[2]
inputs_imp = keras.layers.Input(shape=(past, n_features))
x = tf.keras.layers.GRU(128, return_sequences=True)(inputs_imp)
x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.GRU(64)(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs_imp = tf.keras.layers.Dense(1)(x)

model_imp = keras.Model(inputs=inputs_imp, outputs=outputs_imp)
model_imp.compile(optimizer='adam', loss='mse')
model_imp.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 5, 6)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 5, 128)         │        52,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 89,537 (349.75 KB)

 Trainable params: 89,537 (349.75 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# 6. Entrenamiento
es_imp = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)
history_imp = model_imp.fit( X_train_imp, y_train_imp, epochs=200,
    validation_split=0.2,  shuffle=True, batch_size=64, callbacks=[es_imp])

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.5208 - val_loss: 0.4797
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4684 - val_loss: 0.4649
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4595 - val_loss: 0.4571
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4574 - val_loss: 0.4542
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4586 - val_loss: 0.4440
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4462 - val_loss: 0.4497
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4592 - val_loss: 0.4635
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4511 - val_loss: 0.4572
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4513 - val_loss: 0.4510
Epoch 10/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4478 - val_loss: 0.4644
Epoch 11/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4474 - val_loss: 0.4594
Epoch 12/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4

In [15]:
# 7. Evaluación del modelo con: loss en espacio normalizado y en grados reales (MAE)
results_imp = model_imp.evaluate(X_test_imp, y_test_imp, verbose=1)
print('Test Loss normalizado (MSE):', results_imp)

# 8. Se deshace la normalización para interpretar el error en grados reales (MAE)
temp_std  = scaler.scale_[0]
temp_mean = scaler.mean_[0]
y_pred_norm = model_imp.predict(X_test_imp).flatten()
y_pred_real = y_pred_norm * temp_std + temp_mean
y_test_real = y_test_imp  * temp_std + temp_mean
mae_real = np.mean(np.abs(y_pred_real - y_test_real))
print(f'MAE en grados reales: {mae_real:.3f} °C')

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3646 
Test Loss normalizado (MSE): 0.36457037925720215
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
MAE en grados reales: 1.927 °C


<a name='3.4'></a>
## Cuestión 4: ¿En cuáles de estas aplicaciones se usaría un arquitectura 'many-to-one'?

**a)** Clasificación de sentimiento en textos

**b)** Verificación de voz para iniciar el ordenador.

**c)** Generación de música.

**d)** Un clasificador que clasifique piezas de música según su autor.


**Respuesta: a, b, d**

La arquitectura *many-to-one* recibe una secuencia completa y produce una única salida.

- a) Clasificación de sentimiento: entrada = secuencia de palabras → salida = etiqueta (positivo/negativo)
- b) Verificación de voz: entrada = secuencia de audio → salida = sí/no (es el usuario)
- c) Generación de música: produce una secuencia → arquitectura *one-to-many* o *many-to-many*
- d) Clasificador de música por autor: entrada = secuencia de audio → salida = clase (autor)


<a name='3.5'></a>
## Cuestión 5: ¿Qué ventajas aporta el uso de word embeddings?

**a)** Permiten reducir la dimensión de entrada respecto al one-hot encoding.

**b)** Permiten descubrir la similaridad entre palabras de manera más intuitiva que con one-hot encoding.

**c)** Son una manera de realizar transfer learning en nlp.

**d)** Permiten visualizar las relaciones entre palabras con métodos de reducción de dimensioones como el PCA.


**Respuesta: a, b, c, d** (todas son correctas)

- a) La dimensión del embedding (p. ej. 300) es mucho menor que el vocabulario en one-hot (p. ej. 50 000)
- b) Palabras semánticamente similares quedan cerca en el espacio vectorial, algo imposible con one-hot
- c) Embeddings preentrenados (Word2Vec, GloVe, FastText) transfieren conocimiento lingüístico a nuevas tareas
- d) Al ser vectores densos en baja dimensión, se pueden proyectar y aproximar con PCA y visualizar en ese subespacio.
